##### ktor-client, dataframe, kandy, org.json.XML, org.sqlite.JDBC
* Convert XML data collected via the ktor client to JSON type and load it using DataFrame.readJson.
* Load the SQLite table using DataFrame.readSqlQuery.
* Then, join the two dataframes and create a chart using Kandy.

In [7]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [22]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [30]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(6)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-22 05:02:32, wtch_dt_end:2026-07-22 11:02:32


In [31]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [26]:
@file:DependsOn("org.json:json:20250107")

In [27]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [32]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,535,362,0,0.260000,13,5.308578,2.034965,0.260000,4.560000,5.520000,6.538333,11.343000
rtmWqChpla,Comparable<*>,535,421,0,,44,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,535,1,0,,535,null,null,,,,,
rtmWqWtchStaCd,String,535,14,0,SEA6001,44,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,535,535,0,1,1,268.000000,154.585467,1,134.166667,268.000000,401.833333,535
rtmWqTu,Int,535,115,0,5,62,45.261682,90.166776,1,4.000000,11.000000,34.000000,552
ph,Double,535,126,0,7.340000,30,7.670953,0.358040,6.840000,7.360000,7.570000,8.020000,8.670000
rtmWqSlnty,Float,535,505,0,0.365000,3,20.458748,10.934287,0.019000,8.965833,25.278000,29.472833,32.838001
rtmWqCndctv,Double,535,517,0,0.802000,3,32.039570,16.404196,0.039000,16.302833,37.818001,44.846500,52.540001
rtmWqWtchDtlDt,String,535,46,0,2026-07-22 07:55:00.0,14,null,null,2026-07-22 05:10:00.0,2026-07-22 06:30:00.0,2026-07-22 08:00:00.0,2026-07-22 09:25:00.0,2026-07-22 10:45:00.0


In [33]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) "0" else String.format("%.3f", value.toFloat())

}.convert { rtmWqDoxn and  rtmWqSlnty and rtmWqCndctv and rtmWtchWtem  and ph }.with { String.format("%.3f", it.toFloat()) }


df.schema()

rtmWqDoxn: String
rtmWqChpla: String
rtmWqWtchStaCd: String
num: Int
rtmWqTu: Int
ph: String
rtmWqSlnty: String
rtmWqCndctv: String
rtmWqWtchDtlDt: LocalDateTime
rtmWtchWtem: String

In [34]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")
val renamePairs = df.columnNames().zip(newColunmNames).toTypedArray()
val renamedDf = df.rename(*renamePairs)
renamedDf.columnNames()

[용존산소, 클로로필, 관측정점코드, 순번, 탁도, 수소이온농도, 염분, 전기전도도, 일시, 수온]

In [35]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1,2026-07-22T05:10,6.950,14.990,SEA6001,100,8.070,30.747,47.353,27.210
2,2026-07-22T05:10,0.260,1.020,SEA5002,65,7.270,24.536,38.716,28.480
3,2026-07-22T05:10,5.570,2.122,SEA2007,13,8.070,32.408,52.266,26.130
4,2026-07-22T05:10,5.300,3.270,SEA1301,4,7.740,29.096,43.645,26.180
5,2026-07-22T05:10,6.860,2.880,NEP1002,8,7.600,0.289,0.590,29.180


In [36]:
USE {
    dependencies {
        implementation("org.xerial:sqlite-jdbc:3.49.1.0")
        implementation("ch.qos.logback:logback-classic:1.5.12")
    }
}

In [37]:
import java.sql.Connection
import java.sql.DriverManager

Class.forName("org.sqlite.JDBC")
val connection = DriverManager.getConnection("jdbc:sqlite:/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite")

In [38]:
val sqlStmt = "SELECT * FROM OWQObservatory"
val df_list = DataFrame.readSqlQuery(connection, sqlStmt)
df_list.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
sta_code,String,19,19,0,SEA1002,1,null,null,NEP1001,NEP3001,SEA1301,SEA3003,SEA7002
sta_name,String,19,19,0,시화조력,1,null,null,광양망덕,낙동명지,부산수영,영산목포,천수만
ocean_division,String,19,2,0,특별관리해역,12,null,null,특별관리해역,특별관리해역,특별관리해역,하구 및 만,하구 및 만
lon,Double,19,19,0,126.611000,1,127.588263,1.113453,126.366000,126.540167,127.605000,128.615167,129.387000
lat,Double,19,18,0,35.802000,2,35.687263,0.981615,34.782000,34.990667,35.211000,35.947833,37.731000


In [39]:
val joinedDf = removedDf.join(df_list) { 관측정점코드 match right.sta_code }

In [40]:
joinedDf
    .select{  일시 and 클로로필 and sta_name   }
    .convert{클로로필}.toDouble()
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(클로로필) {axis.name ="클로로필"}
        line{
            color(sta_name){
             //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="gYLUVO" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("gYLUVO");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"sta_name":["새만금","광양초남","천수만","낙동명지","울산매암","시화반월","시화조력","광양망덕","광양적량","영산목포","영산영암","영산영암","낙동명지","시화반월","영산목포","광양초남","새만금","광양망덕","광양적량","광양초남","광양망덕","금강하구","새만금","광양적량","시화반월","낙동명지","영산목포","영산영암","울산매암","시화조력","금강하구","영산영암","광양적량","영산목포","광양초남","시화반월","새만금","울산매암","광양망덕","광양망덕","시화조력","시화반월","울산매암","천수만","새만금","낙동명지","영산목포","광양적량","영산영암","금강하구","광양초남","광양적량","낙동명지","영산목포","광양초남","광양망덕","새만금","시화반월","울산매암","영산영암","울산매암","낙동명지","금강하구","새만금","광양적량","영산영암","광양초남","광양망덕","영산목포","천수만","시화반월","시화조력","울산매암","시화반월","광양망덕","영산영암","광양적량","낙동명지","영산목포","금강하구","광양초남","새만금","낙동명지","천수만","광양적량","새만금","광양초남","금강하구","시화반월","울산매암","광양망덕","시화조력","영산목포","영산영암","시화반월","광양초남","영산영암","영산목포","광양적량","새만금","울산매암","낙동명지","광양망덕","천수만","울산매암","낙동명지","시화조력","광양적량","영산목포","광양망덕","영산영암","광양초남","시화반월","새만금","금강하구","광양망덕","낙동명지","영산영암","영산목포","금강하구","시화반월","광양적량","새만금","울산매암","시화조력","광양망덕","영산영암","새만금","광양초남","낙동명지","천수만","금강하구","광양적량","시화반월","영산목포","영산영암","광양망덕","시화반월","광양적량","새만금","영산목포","낙동명지","광양초남","금강하구","금강하구","광양망덕","새만금","시화반월","울산매암","천수만","시화조력","영산영암","영산목포","광양초남","광양적량","시화반월","광양적량","광양망덕","새만금","금강하구","영산영암","울산매암","울산매암","금강하구","낙동명지","광양초남","새만금","영산영암","시화조력","영산목포","광양망덕","시화반월","광양적량","천수만","새만금","광양초남","울산매암","시화반월","영산영암","낙동명지","금강하구","광양망덕","광양적량","영산목포","영산영암","낙동명지","새만금","광양적량","광양망덕","광양초남","금강하구","시화반월","영산목포","천수만","시화조력","울산매암","시화반월","낙동명지","새만금","영산영암","금강하구","광양적량","광양초남","광양망덕","울산매암","금강하구","광양초남","울산매암","시화조력","영산영암","시화반월","광양적량","낙동명지","영산목포","천수만","광양망덕","새만금","광양망덕","광양초남","영산목포","시화반월","새만금","낙동명지","광양적량","울산매암","영산영암","금강하구","시화조력","광양적량","시화반월","천수만","광양망덕","영산목포","마산봉암","영산영암","새만금","울산매암","광양초남","금강하구","낙동명지","낙동명지","광양적량","영산목포","광양망덕","울산매암","마산봉암","금강하구","영산영암","광양초남","새만금","마산봉암","시화반월","시화조력","광양망덕","새만금","광양적량","울산매암","광양초남","금강하구","낙동명지","천수만","영산영암","영산목포","낙동명지","울산매암","영산목포","새만금","시화반월","광양적량","광양망덕","광양초남","금강하구","마산봉암","영산영암","영산목포","금강하구","울산매암","시화조력","시화반월","천수만","새만금","마산봉암","낙동명지","광양적량","영산영암","광양망덕","광양초남","금강하구","마산봉암","광양망덕","광양적량","새만금","영산영암","광양초남","영산목포","울산매암","낙동명지","광양초남","영산영암","낙동명지","새만금","마산봉암","시화조력","영산목포","광양적량","금강하구","울산매암","시화반월","광양망덕","새만금","광양적량","광양초남","낙동명지","광양망덕","시화반월","영산목포","울산매암","금강하구","영산영암","금강하구","시화반월","시화조력","마산봉암","광양초남","광양적량","울산매암","새만금","영산목포","낙동명지","광양망덕","영산목포","광양초남","낙동명지","광양망덕","금강하구","울산매암","마산봉암","시화반월","광양적량","영산영암","새만금","금강하구","새만금","영산영암","광양망덕","시화반월","시화조력","광양적량","영산목포","울산매암","천수만","낙동명지","광양초남","마산봉암","낙동명지","광양적량","새만금","영산목포","광양초남","광양망덕","영산영암","울산매암","금강하구","시화반월","새만금","금강하구","영산목포","시화반월","천수만","마산봉암","영산영암","광양망덕","광양초남","광양적량","낙동명지","울산매암","영산목포","영산영암","울산매암","마산봉암","금강하구","시화반월","광양초남","낙동명지","광양망덕","광양적량","울산매암","시화반월","낙동명지","새만금","광양초남","금강하구","천수만","광양망덕","마산봉암","영산목포","영산영암","시화조력","광양적량","시화반월","금강하구","낙동명지","마산봉암","광양초남","영산영암","울산매암","새만금","영산목포","광양망덕","천수만","마산봉암","울산매암","낙동명지","광양초남","영산목포","광양망덕","금강하구","광양적량","시화반월","영산영암","새만금","금강하구","낙동명지","영산영암","광양망덕","마산봉암","광양초남","영산목포","시화반월","낙동명지","광양망덕","광양적량","영산영암","마산봉암","천수만","영산목포","새만금"